# Load Capital Markets CSVs → Delta tables

Prereqs: this notebook is attached to a Lakehouse, and the 8 CSV files are in `Files/raw/`.
Run all cells. Idempotent (overwrites tables).

In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
    LongType, DoubleType, DateType, TimestampType)

RAW = "Files/raw"

schemas = {
    "securities": StructType([
        StructField("symbol",    StringType(), False),
        StructField("isin",      StringType(), True),
        StructField("cusip",     StringType(), True),
        StructField("name",      StringType(), True),
        StructField("sector",    StringType(), True),
        StructField("industry",  StringType(), True),
        StructField("exchange",  StringType(), True),
        StructField("currency",  StringType(), True),
        StructField("country",   StringType(), True),
    ]),
    "clients": StructType([
        StructField("client_id",       StringType(), False),
        StructField("name",            StringType(), True),
        StructField("client_type",     StringType(), True),
        StructField("country",         StringType(), True),
        StructField("kyc_tier",        StringType(), True),
        StructField("risk_profile",    StringType(), True),
        StructField("aum_usd",         DoubleType(), True),
        StructField("onboarded_date",  DateType(),   True),
    ]),
    "accounts": StructType([
        StructField("account_id",     StringType(), False),
        StructField("client_id",      StringType(), True),
        StructField("account_type",   StringType(), True),
        StructField("base_currency",  StringType(), True),
        StructField("opened_date",    DateType(),   True),
        StructField("status",         StringType(), True),
    ]),
    "traders": StructType([
        StructField("trader_id", StringType(), False),
        StructField("name",      StringType(), True),
        StructField("desk",      StringType(), True),
        StructField("region",    StringType(), True),
    ]),
    "eod_prices": StructType([
        StructField("symbol",     StringType(), False),
        StructField("trade_date", DateType(),   False),
        StructField("open",       DoubleType(), True),
        StructField("high",       DoubleType(), True),
        StructField("low",        DoubleType(), True),
        StructField("close",      DoubleType(), True),
        StructField("volume",     LongType(),   True),
        StructField("adj_close",  DoubleType(), True),
    ]),
    "trades": StructType([
        StructField("trade_id",   StringType(),    False),
        StructField("trade_ts",   TimestampType(), True),
        StructField("symbol",     StringType(),    True),
        StructField("account_id", StringType(),    True),
        StructField("trader_id",  StringType(),    True),
        StructField("side",       StringType(),    True),
        StructField("quantity",   LongType(),      True),
        StructField("price",      DoubleType(),    True),
        StructField("notional",   DoubleType(),    True),
        StructField("venue",      StringType(),    True),
        StructField("order_type", StringType(),    True),
        StructField("status",     StringType(),    True),
    ]),
    "market_quotes": StructType([
        StructField("symbol",   StringType(),    False),
        StructField("quote_ts", TimestampType(), False),
        StructField("bid",      DoubleType(),    True),
        StructField("ask",      DoubleType(),    True),
        StructField("bid_size", LongType(),      True),
        StructField("ask_size", LongType(),      True),
        StructField("venue",    StringType(),    True),
    ]),
    "positions": StructType([
        StructField("as_of_date",          DateType(),   False),
        StructField("account_id",          StringType(), False),
        StructField("symbol",              StringType(), False),
        StructField("quantity",            LongType(),   True),
        StructField("avg_cost",            DoubleType(), True),
        StructField("market_value_usd",    DoubleType(), True),
        StructField("unrealized_pnl_usd",  DoubleType(), True),
    ]),
}

# Partition large fact tables for performance
partitions = {
    "trades":        [],   # add a derived trade_date col below if you want partitioning
    "market_quotes": [],
    "eod_prices":    [],
}

for name, schema in schemas.items():
    print(f"Loading {name}...")
    df = (spark.read
             .option("header", True)
             .schema(schema)
             .csv(f"{RAW}/{name}.csv"))
    writer = df.write.mode("overwrite").format("delta")
    if partitions.get(name):
        writer = writer.partitionBy(*partitions[name])
    writer.saveAsTable(name)
    print(f"  -> {name}: {df.count():,} rows")

print("\nAll tables loaded.")

## Verify

In [ ]:
spark.sql("SHOW TABLES").show(truncate=False)

In [ ]:
# Quick join sanity check: top traded sectors
spark.sql("""
    SELECT s.sector,
           COUNT(*)              AS trade_count,
           ROUND(SUM(t.notional),2) AS total_notional_usd
    FROM trades t
    JOIN securities s ON s.symbol = t.symbol
    GROUP BY s.sector
    ORDER BY total_notional_usd DESC
""").show(truncate=False)